# Tool-Discovery Agent

Discover bioinformatics tools from papers, inject them into your agent, and run tasks.

**Flow:** Scan agent -> Inject tools -> Run tasks via downstream agent

In [ ]:
import os, sys, subprocess
ST2_DIR = '/content/st2'
if os.path.isdir(os.path.join(ST2_DIR, '.git')):
    subprocess.run(['git', '-C', ST2_DIR, 'pull', '--ff-only'], capture_output=True)
else:
    if os.path.exists(ST2_DIR):
        subprocess.run(['rm', '-rf', ST2_DIR], check=True)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/caixiaoyao2025/st2.git', ST2_DIR], check=True)
ST2_DIR = '/content/st2'
sys.path.insert(0, ST2_DIR)
os.chdir(ST2_DIR)
!pip install -q langchain langchain-openai langgraph pydantic graph-tool-call 2>/dev/null || true
from IPython.display import display, Markdown
display(Markdown('**Setup done**'))

## Step 1 - Input your agent & API key

In [ ]:
import os, subprocess
from IPython.display import display, Markdown
import ipywidgets as widgets

# ---- Known agent mappings: method + register method ----
KNOWN_AGENTS = {
    'biomni': {'method': 'go', 'register': 'add_tool'},
    'cellagent': {'method': 'run', 'register': 'add_tool'},
    'geneagent': {'method': 'run', 'register': 'add_tool'},
    'crispr': {'method': 'run', 'register': 'add_tool'},
    'biochatter': {'method': 'run', 'register': 'add_tool'},
    'langchain': {'method': 'invoke', 'register': 'add_tool'},
    'smolagents': {'method': 'run', 'register': 'add_tool'},
    'dspy': {'method': 'forward', 'register': 'add_tool'},
    'crewai': {'method': 'kickoff', 'register': 'add_tool'},
    'metagpt': {'method': 'run', 'register': 'add_tool'},
}
PROBE_ORDER = ['go', 'run', 'execute', 'predict', 'forward', 'invoke']

path_input = widgets.Text(
    value='https://github.com/snap-stanford/Biomni.git',
    placeholder='Git URL or /content/my_agent',
    description='Agent:', layout=widgets.Layout(width='90%'))
key_input = widgets.Password(
    value='', placeholder='ark-... or sk-...',
    description='API Key:', layout=widgets.Layout(width='90%'))
model_input = widgets.Text(
    value='deepseek-v4-flash-ga-260731',
    description='Model:', layout=widgets.Layout(width='70%'))
base_input = widgets.Text(
    value='https://ark.cn-beijing.volces.com/api/v3',
    description='Base URL:', layout=widgets.Layout(width='90%'))
btn = widgets.Button(description='Connect agent', button_style='primary')
out = widgets.Output()

def on_connect(_):
    out.clear_output()
    with out:
        agent_src = path_input.value.strip()
        api_key = key_input.value.strip()
        model = model_input.value.strip()
        base_url = base_input.value.strip()
        if not agent_src:
            display(Markdown('**ERROR:** Enter agent path or URL')); return
        if not api_key:
            display(Markdown('**ERROR:** Enter API key')); return
        if agent_src.startswith('http'):
            agent_dir = '/content/_agent_' + agent_src.split('/')[-1].replace('.git', '')
            if not os.path.isdir(agent_dir):
                subprocess.run(['git', 'clone', '--depth', '1', agent_src, agent_dir],
                               capture_output=True, text=True)
            display(Markdown(f'Cloned to `{agent_dir}`'))
        else:
            agent_dir = agent_src
            if not os.path.isdir(agent_dir):
                display(Markdown(f'**ERROR:** `{agent_dir}` not found')); return
        os.environ['OPENAI_API_KEY'] = api_key
        os.environ['OPENAI_BASE_URL'] = base_url
        os.environ['OPENAI_MODEL'] = model
        os.environ['WESTLAKE_API_KEY'] = api_key
        os.environ['BIOMNI_SOURCE'] = 'Custom'
        os.environ['BIOMNI_LLM'] = model
        os.environ['BIOMNI_CUSTOM_BASE_URL'] = base_url
        os.environ['BIOMNI_CUSTOM_API_KEY'] = api_key
        get_ipython().user_ns['agent_dir'] = agent_dir
        get_ipython().user_ns['api_key_val'] = api_key
        get_ipython().user_ns['model_val'] = model
        get_ipython().user_ns['base_url_val'] = base_url
        display(Markdown(f'**Agent:** `{agent_dir}` | **Model:** `{model}` | **Key:** `{api_key[:8]}...`'))

btn.on_click(on_connect)
display(widgets.VBox([path_input, key_input, model_input, base_input, btn, out]))

## Step 2 - Scan agent & detect wiring

In [ ]:
import os, sys, json
from IPython.display import display, Markdown
import yaml

from agent_connector.scanner import build_schema

schema = build_schema(agent_dir, include_evidence=False)
detected = [
    '**Detected:**',
    '- agent_class = `' + str(schema.get('agent_class', 'N/A')) + '`',
    '- module_path = `' + str(schema.get('module_path', 'N/A')) + '`',
    '- registration_method = `' + str(schema.get('registration_method', 'N/A')) + '`',
    '- wiring_style = `' + str(schema.get('wiring_style', 'N/A')) + '`',
    '- init_signature = `' + str(schema.get('init_signature', 'N/A')) + '`',
]
display(Markdown('<br>'.join(detected)))

reg_path = os.path.join(ST2_DIR, 'data', 'mcp_registry.yaml')
tools = yaml.safe_load(open(reg_path, encoding='utf-8'))['tools']
display(Markdown(f'**Registry:** {len(tools)} tools'))

## Step 3 - Resolve execution method

In [ ]:
from IPython.display import display, Markdown

agent_class = (schema.get('agent_class') or '').lower()

# Layer 1: known agent -> direct mapping
exec_method = None
for pat, info in KNOWN_AGENTS.items():
    if pat in agent_class or pat in agent_dir.lower():
        exec_method = info['method']
        display(Markdown(f'**Known agent** `{pat}` -> `{exec_method}`'))
        break

# Layer 2: scan source for .go()/.run() hints
if exec_method is None:
    hits = {}
    for root, dirs, files in os.walk(agent_dir):
        dirs[:] = [d for d in dirs if d not in ('.git','__pycache__','node_modules','.venv')]
        for f in files:
            if f.endswith('.py'):
                try:
                    text = open(os.path.join(root,f), encoding='utf-8', errors='replace').read()
                except: continue
                for m in PROBE_ORDER:
                    hits[m] = hits.get(m, 0) + text.count(f'.{m}(')
    if any(v > 0 for v in hits.values()):
        exec_method = max(hits, key=hits.get)
        display(Markdown(f'**Source hint** -> `{exec_method}` ({hits[exec_method]} occurrences)'))

# Layer 3: default
if exec_method is None:
    exec_method = 'run'
    display(Markdown('**No signal found.** Defaulting to `run`.'))

get_ipython().user_ns['exec_method_override'] = exec_method

## Step 4 - Preflight check & install

In [ ]:
import os
from IPython.display import display, Markdown
from agent_connector.agent_preflight import preflight, preflight_report

pf = preflight(agent_dir, agent_module_path=schema.get('module_path'))
display(Markdown(preflight_report(pf)))

In [ ]:
import subprocess, os, sys
from IPython.display import display, Markdown

if pf.status == 'SETUP_REQUIRED':
    pkgs = pf.pip_installable
    if pkgs:
        display(Markdown(f'Installing **{len(pkgs)}** packages...'))
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        display(Markdown(f'**Done** (returncode={r.returncode})'))
    else:
        display(Markdown('No auto-installable packages'))
else:
    display(Markdown(f'Status = `{pf.status}`'))

## Step 5 - Create agent & inject tools

In [ ]:
import os, sys, json, importlib
from IPython.display import display, Markdown

from agent_connector.generator import generate_wiring, load_wrappers, load_adapter

schema['execution_method'] = exec_method_override

# Force-set BIOMNI env vars BEFORE any agent import (so default_config picks them up)
import os as _os
_m = _os.environ.get('OPENAI_MODEL', '')
_k = _os.environ.get('OPENAI_API_KEY') or _os.environ.get('WESTLAKE_API_KEY')
_u = _os.environ.get('OPENAI_BASE_URL', '')
if _m and not _os.environ.get('BIOMNI_LLM'): _os.environ['BIOMNI_LLM'] = _m
if _k and not _os.environ.get('BIOMNI_CUSTOM_API_KEY'): _os.environ['BIOMNI_CUSTOM_API_KEY'] = _k
if _u and not _os.environ.get('BIOMNI_CUSTOM_BASE_URL'): _os.environ['BIOMNI_CUSTOM_BASE_URL'] = _u
if not _os.environ.get('BIOMNI_SOURCE'): _os.environ['BIOMNI_SOURCE'] = 'Custom'

# Generate wiring
wiring_dir = os.path.join(ST2_DIR, 'wiring')
wiring = generate_wiring(tools, schema, out_dir=wiring_dir)
display(Markdown('Wiring mode: `' + wiring['mode'] + '`'))

# Load wrappers as functions (Biomni needs inspect.getsource)
sys.path.insert(0, wiring_dir)
sys.path.insert(0, agent_dir)
wrappers = load_wrappers(package_name='generated_tools', registration_style='function')
display(Markdown(f'**{len(wrappers)} wrappers loaded**'))

# Create agent via adapter
agent = None
adapter = None

if schema.get('module_path') or schema.get('registration_method'):
    try:
        adapter = load_adapter(schema.get('agent_class') or 'Agent',
                               adapter_path=wiring['artifacts']['adapter'])
        agent = adapter.create_agent(use_tool_retriever=False)
        display(Markdown(f'**Agent created via adapter:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'`create_agent()` failed: `{e}`'))

# Fallback: if adapter.create_agent() failed (e.g. react needs all Biomni deps), try A1
# A1 is lighter — it doesn't load all built-in tools in __init__
if agent is None and 'biomni' in agent_dir.lower():
    try:
        sys.path.insert(0, agent_dir)
        from biomni.agent.a1 import A1
        agent = A1(
            path=os.path.join(ST2_DIR, '_agent_data'),
            llm=os.environ.get('OPENAI_MODEL', 'gpt-4o'),
            source='Custom',
            base_url=os.environ.get('OPENAI_BASE_URL'),
            api_key=os.environ.get('OPENAI_API_KEY'),
            use_tool_retriever=False,
            expected_data_lake_files=[],
        )
        display(Markdown(f'**A1 agent created:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'A1 fallback failed: `{e}`'))

# Fallback: DynamicAgent
if agent is None:
    class DynamicAgent:
        def __init__(self): self.tools = []
    DynamicAgent.add_tool = lambda self, t: self.tools.append(t)
    agent = DynamicAgent()
    display(Markdown(f'**Fallback:** DynamicAgent'))

# Inject tools: create LangChain tools from _TOOL_SPEC and append to agent.tools
from langchain_core.tools import tool as lc_tool
from agent_connector.tool_runner import run_tool_spec, format_result

if not hasattr(agent, 'tools') or agent.tools is None:
    agent.tools = []

injected = 0
for w in wrappers:
    ts = getattr(w, '_TOOL_SPEC', None)
    if ts is None and hasattr(w, '__module__'):
        mod = importlib.import_module(w.__module__)
        ts = getattr(mod, '_TOOL_SPEC', None)
    if ts:
        def _make_runner(spec=ts):
            _desc = spec.get('description', spec['name'])
            # Build input schema string for LLM
            _inputs = spec.get('inputs') or {}
            if _inputs:
                _params = []
                for _pname, _pmeta in _inputs.items():
                    _req = 'required' if (_pmeta or {}).get('required') else 'optional'
                    _ptype = (_pmeta or {}).get('type', 'string')
                    _pdesc = (_pmeta or {}).get('description', '')
                    _params.append(f'  {_pname} ({_ptype}, {_req}): {_pdesc}')
                _desc += '\nParameters:\n' + chr(10).join(_params)
            def fn(**kwargs):
                return format_result(run_tool_spec(spec, dict(kwargs)))
            fn.__name__ = spec['name']
            fn.__doc__ = _desc
            return lc_tool(fn)
        agent.tools.append(_make_runner())
        injected += 1
    else:
        agent.tools.append(w)
        injected += 1

display(Markdown(f'**{injected} tools injected** into `{type(agent).__name__}.tools`'))

# Show available tool names
tool_names = []
for t in agent.tools:
    name = getattr(t, 'name', None) or getattr(t, '__name__', '?')
    tool_names.append(name)
display(Markdown('**Available tools:** ' + ', '.join(tool_names)))

# Verify exec method
is_known = any(pat in agent_dir.lower() or pat in type(agent).__name__.lower()
               for pat in KNOWN_AGENTS)
if is_known:
    display(Markdown(f'**Known agent** -> method `{exec_method_override}` (trusted)'))
else:
    probe = None
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent): probe = m; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            probe = m; break
    if probe:
        display(Markdown(f'**Probe OK:** `agent.{probe}()` exists'))
        get_ipython().user_ns['exec_method_override'] = probe
    else:
        tried = ', '.join(f'`{m}()`' for m in PROBE_ORDER)
        display(Markdown(f'**Probe FAILED:** tried {tried}'))
        display(Markdown('Go to **Step 5b** to provide your agent init code manually.'))

get_ipython().user_ns['agent'] = agent
get_ipython().user_ns['wrappers'] = wrappers
get_ipython().user_ns['adapter'] = adapter

# --- LangChain agent wrapper (tool calling loop) ---
# Wraps injected tools so the LLM selects them via function calling
# instead of code execution.
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.prebuilt import create_react_agent

if agent.tools:
    lc_llm = ChatOpenAI(
        model=os.environ.get('MODEL_ID', 'deepseek-v4-flash-ga-260731'),
        base_url=os.environ.get('WESTLAKE_BASE_URL', 'https://ark.cn-beijing.volces.com/api/v3'),
        api_key=os.environ.get('WESTLAKE_API_KEY', 'dummy'),
        temperature=0,
    )
    lc_agent = create_react_agent(
        model=lc_llm,
        tools=agent.tools,
        prompt='You are a bioinformatics assistant. Use the provided tools to answer questions. Always call a tool first before answering.',
    )
    display(Markdown(f'**LangChain agent** created with {len(agent.tools)} tools (tool calling loop)'))
else:
    lc_agent = None
    display(Markdown('**No tools injected** - LangChain agent not created'))

get_ipython().user_ns['lc_agent'] = lc_agent

## Step 5b - Manual agent init (only if Step 5 failed)

If Step 5 succeeded, **skip this cell**.

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets

already_ok = is_known
if not already_ok:
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent): already_ok = True; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            already_ok = True; break

if already_ok:
    display(Markdown('Step 5 succeeded. **Skipping.**'))
else:
    display(Markdown('### Agent execution method not recognized\n\n'
        '**Tried:** ' + ', '.join(f'`{m}`' for m in PROBE_ORDER) + '\n\n'
        '**Please paste your agent init code below:**'))
    code_area = widgets.Textarea(
        value='# from my_agent import MyAgent\n# agent = MyAgent(model="gpt-4")\n# agent.add_tools(wrappers)\n',
        placeholder='agent = MyAgent(...)',
        layout=widgets.Layout(width='90%', height='150px'))
    method_input = widgets.Dropdown(
        options=['run', 'execute', 'go', 'predict', 'forward', 'invoke', '__call__'],
        value='run', description='Exec method:', layout=widgets.Layout(width='60%'))
    apply_btn = widgets.Button(description='Apply', button_style='primary')
    out = widgets.Output()

    def on_apply(_):
        out.clear_output()
        with out:
            try:
                local_ns = {'wrappers': wrappers, 'agent_dir': agent_dir}
                exec(code_area.value, local_ns)
                new_agent = local_ns.get('agent')
                if new_agent is None:
                    display(Markdown('Error: no `agent` variable found.')); return
                method = method_input.value
                fn = getattr(new_agent, method, None) if method != '__call__' else new_agent
                if fn is None or not callable(fn):
                    display(Markdown(f'Error: `{method}` not callable.')); return
                get_ipython().user_ns['agent'] = new_agent
                get_ipython().user_ns['exec_method_override'] = method
                agent = new_agent
                display(Markdown(f'**Done!** Agent=`{type(new_agent).__name__}` method=`{method}`'))
            except Exception as e:
                display(Markdown(f'Error: `{e}`'))

    apply_btn.on_click(on_apply)
    display(widgets.VBox([code_area, method_input, apply_btn, out]))

## Step 6 - Run tasks via downstream agent

Your query is sent to the agent via `agent.{method}(query)`. The agent uses the injected tools internally.

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets
import time

# Prefer LangChain agent (tool calling loop) over downstream agent (code execution loop)
use_lc = lc_agent is not None
if use_lc:
    display(Markdown('**Mode:** LangChain agent (tool calling loop)'))
else:
    display(Markdown(f'**Mode:** `{type(agent).__name__}.{method}()` (code execution loop)'))

# Dynamically build tool list from injected tools
_tool_list = []
for t in agent.tools:
    _name = getattr(t, 'name', None) or getattr(t, '__name__', '?')
    _desc = getattr(t, 'description', '') or ''
    _tool_list.append(f'- {_name}: {_desc[:80]}')
_tools_block = chr(10).join(_tool_list) if _tool_list else '(no tools injected)'

query_input = widgets.Text(
    value=f'Look up the human BRCA1 gene sequence (ENSG00000012048) from Ensembl, then compute its reverse complement.',
    placeholder='Ask a bioinformatics question...',
    description='Query:', layout=widgets.Layout(width='95%'))
run_btn = widgets.Button(description='Run agent', button_style='success')
result_out = widgets.Output()

def run_query(_):
    result_out.clear_output()
    with result_out:
        query = query_input.value.strip()
        if not query:
            display(Markdown('**Enter a query**')); return
        display(Markdown(f'**Query:** {query}'))
        t0 = time.time()
        try:
            if use_lc:
                display(Markdown('**Calling:** `lc_agent.invoke()` ...'))
                result = lc_agent.invoke({'messages': [{'role': 'user', 'content': query}]})
                # Extract final message
                msgs = result.get('messages', [])
                final = msgs[-1].content if msgs else str(result)
            else:
                display(Markdown(f'**Calling:** `agent.{method}(query)` ...'))
                final = exec_fn(query)
            elapsed = time.time() - t0
            display(Markdown(f'**Done** ({elapsed:.1f}s)'))
            display(Markdown(f'### Result\n{str(final)[:3000]}'))
        except Exception as e:
            display(Markdown(f'**Error:** `{type(e).__name__}: {e}`'))

run_btn.on_click(run_query)
display(widgets.VBox([query_input, run_btn, result_out]))